In [0]:
path = "s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_business.json"
df = spark.read.json(path)
display(df.columns)

_1
address
attributes
business_id
categories
city
hours
is_open
latitude
longitude
name


In [0]:
total_rows = df.count()
total_rows

150346

In [0]:
# Summary statistics
display(df.describe())

# Null value counts and percentages per column
from pyspark.sql.functions import col, count, isnan, when

total_count = df.count()
# Get numeric column types for isnan check
numeric_types = {'double', 'float'}
df_schema = {field.name: field.dataType.typeName() for field in df.schema.fields}

null_stats = (
    df.select([
        count(when(
            col(c).isNull() | (isnan(col(c)) if df_schema[c] in numeric_types else False), 
            c
        )).alias(f"{c}_null_count")
        for c in df.columns
    ])
)

null_counts = null_stats.collect()[0].asDict()
null_percentages = {col: (count / total_count) * 100 for col, count in null_counts.items()}

# Display null counts and percentages as a DataFrame
import pandas as pd

summary_data = [
    {"column": col.replace("_null_count", ""), "null_count": null_counts[col], "null_percentage": null_percentages[col]}
    for col in null_counts
]
display(pd.DataFrame(summary_data))

summary,address,business_id,categories,city,is_open,latitude,longitude,name,postal_code,review_count,stars,state
count,150346,150346,150243,150346,150346,150346,150346,150346,150346,150346,150346,150346
mean,7369.333333333333,null,null,null,0.7961502135075094,36.67115006414568,-89.35733948971439,1252.4,45177.81755426108,44.86656113232144,3.5967235576603303,null
stddev,8738.777641447725,null,null,null,0.4028599390900635,5.872758917014048,14.918501679930612,811.1275005954502,26395.88208585646,121.12013570117074,0.9744207509201362,null
min,,---kPU91CF4Lq2-WlRu9Lw,"3D Printing, Local Services, Hobby Shops, Shopping",AB Edmonton,0,27.555127,-120.095137,Grow Academy,,5,1.0,AB
max,​185 E State St,zzyx5x0Z7xXWWvWnZFuxlQ,"Zoos, Tours, Arts & Entertainment, Hotels & Travel, Active Life",​Lithia,1,53.6791969,-73.2004570502,​​Transformational Abdominal Massage by Jada Delaney,T9E 0V3,7568,5.0,XMS


column,null_count,null_percentage
address,0,0.0
attributes,13744,9.14158008859564
business_id,0,0.0
categories,103,0.06850864007023798
city,0,0.0
hours,23223,15.446370372341134
is_open,0,0.0
latitude,0,0.0
longitude,0,0.0
name,0,0.0


In [0]:
count_null_categories_attributes = df.filter(
    col("categories").isNull() & col("attributes").isNull()
).count()
count_null_categories_attributes

102

In [0]:
from pyspark.sql.functions import approx_count_distinct

unique_counts = df.select([
    approx_count_distinct(col(c)).alias(f"{c}_unique_count") for c in df.columns
])

unique_counts_pd = unique_counts.toPandas().T.reset_index()
unique_counts_pd.columns = ['column', 'unique_count']
unique_counts_pd['column'] = unique_counts_pd['column'].str.replace('_unique_count', '')

display(unique_counts_pd)

column,unique_count
address,125266
attributes,59541
business_id,154923
categories,80389
city,1373
hours,49829
is_open,2
latitude,135015
longitude,127115
name,111923


In [0]:
display(df.sample(False, 0.01).limit(20))

address,attributes,business_id,categories,city,hours,is_open,latitude,longitude,name,postal_code,review_count,stars,state
3001 Highway 31 W,"List(null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1, null, null, null, null, null, null)",sqSqqLy0sN8n2IZrAbzidQ,"Pizza, Chicken Wings, Sandwiches, Restaurants",White House,"List(10:0-1:0, 10:0-0:0, 10:0-1:0, 10:0-0:0, 10:0-0:0, 10:0-0:0, 10:0-0:0)",1,36.4647467622,-86.6591868908,Domino's Pizza,37188,8,3.5,TN
100 Iberville St,"List(null, null, null, null, null, null, null, null, null, True, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2, null, null, null, null, null, 'no')",w_AMNoI1iG9eay7ncmc67w,"Event Planning & Services, Hotels, Hotels & Travel",New Orleans,null,1,29.951359,-90.0646715,River 127,70130,12,3.0,LA
1901 E Washington St,"List(null, null, null, null, null, null, null, True, null, True, {'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}, False, null, null, null, null, null, null, null, True, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null)",xAAhPuZ8FG6OlvAw5m1BVQ,"Active Life, Playgrounds, Parks, Swimming Pools",Indianapolis,null,1,39.7667851,-86.12748,Willard Park,46201,5,3.5,IN
216 W Beidler Rd,"List(null, null, null, null, null, null, null, True, null, True, {'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}, null, null, null, null, null, True, null, null, null, null, null, null, null, null, null, null, null, null, null, False, null, 1, null, null, True, null, null, null)",pmuuoDqNZp7518AUd-YmPA,"Restaurants, Bakeries, Caterers, Bagels, Food, Event Planning & Services",King of Prussia,"List(7:0-14:0, null, 7:0-14:0, 8:0-13:0, 7:0-13:0, 7:0-13:0, 7:0-13:0)",1,40.1124813,-75.3799748,Bagelicious - King Of Prussia,19406,60,3.5,PA
518 Port Royal Ave,"List(null, null, null, null, null, null, null, True, null, True, {'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}, null, null, null, null, null, null, True, null, null, null, null, null, null, null, null, null, null, null, null, True, null, 1, null, null, False, null, True, null)",jL_NufxqXi-BpW5uXKsPwQ,"Drugstores, Pharmacy, Health & Medical, Convenience Stores, Food, Shopping",Philadelphia,"List(8:0-23:0, 0:0-0:0, 8:0-23:0, 8:0-22:0, 8:0-22:0, 8:0-22:0, 8:0-22:0)",1,40.0612774187,-75.2372046062,CVS Pharmacy,19128,11,3.0,PA
"252 W Swamp Rd, Ste 56","List(null, null, null, null, null, null, null, True, False, True, {'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}, True, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 3, null, null, null, null, True, null)",vSE6lkp5FBcttPxEbQ9XCw,"Health & Medical, Medical Spas, Beauty & Spas, Skin Care, Makeup Artists",Doylestown,"List(9:0-15:0, 10:0-17:0, null, null, null, 10:0-17:0, 9:0-17:0)",1,40.3311267851,-75.1371342508,Kim Kelly & Co Clinical Skin Care,18901,7,5.0,PA
5500 Gulf Blvd,"List(null, null, null, null, null, null, null, null, null, True, {'garage': False, 'street': False, 'validated': False, 'lot': False, 'valet': False}, True, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 3, null, null, null, null, null, null)",vweEeZwiaRRWX2Cu9Iq66Q,"Day Spas, Massage, Beauty & Spas",St. Pete Beach,null,1,27.7305212,-82.7452183,Bodyworks Salon & Day Spa,33706,6,2.5,FL
"1990 Marlton Pike E, Ste 10","List(null, null, u'none', {'touristy': False, 'hipster': False, 'romantic': False, 'divey': False, 'intimate': False, 'trendy': False, 'upscale': False, 'classy': True, 'casual': True}, null, null, null, True, False, True, {'garage':

In [0]:
from pyspark.sql.functions import col

# Flatten the hours column: create a DataFrame with business_id and each day as a column
df_hours = df.select(
    col("business_id"),
    col("hours.Monday").alias("Monday"),
    col("hours.Tuesday").alias("Tuesday"),
    col("hours.Wednesday").alias("Wednesday"),
    col("hours.Thursday").alias("Thursday"),
    col("hours.Friday").alias("Friday"),
    col("hours.Saturday").alias("Saturday"),
    col("hours.Sunday").alias("Sunday")
)

display(df_hours.limit(30))

business_id,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Pns2l4eNsfO8kk83dixA6A,null,null,null,null,null,null,null
mpf3x-BjTdTEA3yCZrAYPw,0:0-0:0,8:0-18:30,8:0-18:30,8:0-18:30,8:0-18:30,8:0-14:0,null
tUFrWirKiKi_TAnsVWINQQ,8:0-22:0,8:0-22:0,8:0-22:0,8:0-22:0,8:0-23:0,8:0-23:0,8:0-22:0
MTSW4McQd7CbVtyjqoe9mw,7:0-20:0,7:0-20:0,7:0-20:0,7:0-20:0,7:0-21:0,7:0-21:0,7:0-21:0
mWMc6_wTdE0EUBKIGXDVfA,null,null,14:0-22:0,16:0-22:0,12:0-22:0,12:0-22:0,12:0-18:0
CF33F8-E6oudUQ46HnavjQ,0:0-0:0,6:0-22:0,6:0-22:0,6:0-22:0,9:0-0:0,9:0-22:0,8:0-22:0
n_0UpQx1hsNbnPUSlodU8w,0:0-0:0,10:0-18:0,10:0-18:0,10:0-18:0,10:0-18:0,10:0-18:0,12:0-18:0
qkRM_2X51Yqxk3btlwAQIg,9:0-17:0,9:0-17:0,9:0-17:0,9:0-17:0,9:0-17:0,null,null
k0hlBqXX-Bt0vf1op7Jr1w,null,null,null,null,null,null,null
bBDDEgkFA1Otx9Lfe7BZUQ,0:0-0:0,6:0-21:0,6:0-21:0,6:0-16:0,6:0-16:0,6:0-17:0,6:0-21:0


In [0]:
df_hours_count = df_hours.count()
df_hours_count

150346

In [0]:
from pyspark.sql.functions import col

# List of hour columns (excluding business_id)
hour_columns = [c for c in df_hours.columns if c != "business_id"]

# Count rows with null in each hour column (excluding business_id)
null_counts = {
    col_name: df_hours.filter(col(col_name).isNull()).count()
    for col_name in hour_columns
}

null_counts

{'Monday': 35872,
 'Tuesday': 29715,
 'Wednesday': 26575,
 'Thursday': 25148,
 'Friday': 25347,
 'Saturday': 39576,
 'Sunday': 69174}

In [0]:
from pyspark.sql.functions import col

hour_columns = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

count_all_null = df_hours.filter(
    sum([col(c).isNull().cast("int") for c in hour_columns]) == len(hour_columns)
).count()

print(count_all_null)

23223


In [0]:
from pyspark.sql.functions import col

# Flatten the attributes column: create a DataFrame with business_id and each attribute as a column
attribute_keys = [field.name for field in df.schema["attributes"].dataType.fields]

df_attributes = df.select(
    col("business_id"),
    *[col(f"attributes.{key}").alias(key) for key in attribute_keys]
)

display(df_attributes.limit(30))

business_id,AcceptsInsurance,AgesAllowed,Alcohol,Ambience,BYOB,BYOBCorkage,BestNights,BikeParking,BusinessAcceptsBitcoin,BusinessAcceptsCreditCards,BusinessParking,ByAppointmentOnly,Caters,CoatCheck,Corkage,DietaryRestrictions,DogsAllowed,DriveThru,GoodForDancing,GoodForKids,GoodForMeal,HairSpecializesIn,HappyHour,HasTV,Music,NoiseLevel,Open24Hours,OutdoorSeating,RestaurantsAttire,RestaurantsCounterService,RestaurantsDelivery,RestaurantsGoodForGroups,RestaurantsPriceRange2,RestaurantsReservations,RestaurantsTableService,RestaurantsTakeOut,Smoking,WheelchairAccessible,WiFi
Pns2l4eNsfO8kk83dixA6A,null,null,null,null,null,null,null,null,null,null,null,True,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
mpf3x-BjTdTEA3yCZrAYPw,null,null,null,null,null,null,null,null,null,True,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
tUFrWirKiKi_TAnsVWINQQ,null,null,null,null,null,null,null,True,null,True,"{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}",False,False,False,null,null,False,null,null,null,null,null,False,False,null,null,null,False,null,null,False,null,2,False,null,False,null,True,u'no'
MTSW4McQd7CbVtyjqoe9mw,null,null,u'none',null,null,null,null,True,null,False,"{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}",False,True,null,null,null,null,null,null,null,null,null,null,null,null,null,null,False,null,null,False,null,1,null,null,True,null,null,u'free'
mWMc6_wTdE0EUBKIGXDVfA,null,null,null,null,null,null,null,True,null,True,"{'garage': None, 'street': None, 'validated': None, 'lot': True, 'valet': False}",null,False,null,null,null,null,null,null,True,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,True,null,True,null
CF33F8-E6oudUQ46HnavjQ,null,null,u'none',None,null,null,null,False,null,True,None,False,False,False,null,null,False,True,null,True,null,null,False,True,null,null,null,True,u'casual',null,True,True,1,False,False,True,null,True,u'no'
n_0UpQx1hsNbnPUSlodU8w,null,null,null,null,null,null,null,True,null,True,"{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2,null,null,null,null,null,null
qkRM_2X51Yqxk3btlwAQIg,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
k0hlBqXX-Bt0vf1op7Jr1w,null,null,u'full_bar',"{'romantic': False, 'intimate': False, 'touristy': False, 'hipster': False, 'divey': False, 'classy': False, 'trendy': False, 'upscale': False, 'casual': False}",null,null,null,null,null,True,"{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}",null,True,null,null,null,null,null,null,True,null,null,null,True,null,u'average',null,True,u'casual',null,False,True,1,False,null,True,null,null,u'free'
bBDDEgkFA1Otx9Lfe7BZUQ,null,null,u'none',null,null,null,null,null,null,True,"{'garage': False, 'street': False, 'validated': False, 'lot': False, 'valet': False}",False,False,False,null,null,False,True,null,True,null,null,False,True,null,null,null,True,'casual',null,True,False,1,False,False,True,null,True,u'no'


In [0]:
from pyspark.sql.functions import col, count, isnan, when, approx_count_distinct
import pandas as pd

# List of column names
attribute_columns = df_attributes.columns

# Total number of rows
total_count = df_attributes.count()

# Identify numeric columns (required for isnan())
numeric_types = {'double', 'float'}
df_attr_schema = {
    field.name: field.dataType.typeName()
    for field in df_attributes.schema.fields
}

# Count null values in each column
null_stats = df_attributes.select([
    count(
        when(
            col(c).isNull() |
            (isnan(col(c)) if df_attr_schema[c] in numeric_types else False),
            c
        )
    ).alias(f"{c}_null_count")
    for c in attribute_columns
])

# Count approximate unique values in each column
unique_stats = df_attributes.select([
    approx_count_distinct(col(c)).alias(f"{c}_unique_count")
    for c in attribute_columns
])

# Collect results
null_counts = null_stats.collect()[0].asDict()
unique_counts = unique_stats.collect()[0].asDict()

# Calculate filled (non-null) percentage
filled_percentages = {
    c: ((total_count - null_counts[f"{c}_null_count"]) / total_count) * 100
    for c in attribute_columns
}

# Create summary DataFrame
summary_data = [
    {
        "column": c,
        "null_count": null_counts[f"{c}_null_count"],
        "filled_percentage": round(filled_percentages[c], 2),
        "unique_count": unique_counts[f"{c}_unique_count"]
    }
    for c in attribute_columns
]

summary_df = pd.DataFrame(summary_data)

# Display the summary
display(summary_df)

column,null_count,filled_percentage,unique_count
business_id,0,100.0,154923
AcceptsInsurance,144633,3.8,3
AgesAllowed,150217,0.09,3
Alcohol,107157,28.73,7
Ambience,106067,29.45,2334
BYOB,145895,2.96,3
BYOBCorkage,148902,0.96,7
BestNights,144652,3.79,123
BikeParking,77708,48.31,3
BusinessAcceptsBitcoin,132916,11.59,3
